In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, minmax_scale
from sklearn.metrics import root_mean_squared_error

In [2]:

# Step 1: Load filtered test and training data
test_df = pd.read_csv('filtered_test_data.csv')
train_df = pd.read_csv('filtered_train_data.csv')

# Step 2: Prepare training data
X_train = pd.DataFrame(train_df[['horse_rating', 'declared_weight', 'actual_weight',
                    'win_odds', 'distance', 'race_class','finish_time']])

# Step 3: Prepare test data
X_test = pd.DataFrame(test_df[['horse_rating', 'declared_weight', 'actual_weight',
                  'win_odds', 'distance', 'race_class','finish_time']])

# Step 4: Standardize features using training data stats
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),columns=X_test.columns)

#grab target values and drop from
y_test = X_test_scaled["finish_time"]
y_train = X_train_scaled["finish_time"]
X_train_scaled = X_train_scaled.drop(columns="finish_time")
X_test_scaled = X_test_scaled.drop(columns="finish_time")
# Step 5: Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Step 6: Predict finish_time on test set
#test_df['predicted_finish_time'] = model.predict(X_test_scaled)
predicted  = model.predict(X_test_scaled)

# Step 7: Group by race_id and check if the predicted winner matches the actual winner
def predict_winner(group):
    predicted_winner_index = group['predicted_finish_time'].idxmin()
    return group.loc[predicted_winner_index, 'won'] == 1

# Apply prediction accuracy per race
#results = test_df.groupby('race_id').apply(predict_winner)

# Step 8: Calculate accuracy
#accuracy = results.mean()
#print(f"Prediction Accuracy: {accuracy:.4f} ({results.sum()} correct out of {len(results)} races)")


In [3]:
rmse = root_mean_squared_error(y_test,predicted)
round(rmse,4)

0.0759

In [4]:
round(model.score(X_test_scaled,y_test),4)

0.9942